In [151]:
import sys 
import os
from pathlib import Path
import pandas as pd

#from word_embeddings import *
import torch

In [152]:
from csv import QUOTE_NONE
import sys
import csv

maxInt = sys.maxsize

while True:
    # decrease the maxInt value by factor 10 
    # as long as the OverflowError occurs.

    try:
        csv.field_size_limit(maxInt)
        break
    except OverflowError:
        maxInt = int(maxInt/10)


base_path = Path(os.path.abspath("")).parents[1] / "dataset_creation" / "data"
datasets = {
    "school_shooters": base_path / "school_shooters.csv",
    "manifestos": base_path / "manifestos.csv",
    "stair_twitter_archive": base_path / "stair_twitter_archive.csv",
    "twitter": base_path / "twitter.csv",
    "stream_of_consciousness": base_path / "stream_of_consciousness.csv"
}

schoolshootersinfo_df = pd.read_csv(datasets["school_shooters"], encoding="utf-8", delimiter="‎", engine="python", quoting=QUOTE_NONE)
#manifesto_df = pd.read_csv(datasets["manifestos"], encoding="utf-8", delimiter="‎", engine="python", quoting=QUOTE_NONE)
stair_twitter_archive_df = pd.read_csv(datasets["stair_twitter_archive"], encoding="utf-8", delimiter="‎", engine="python", quoting=QUOTE_NONE)
twitter_df = pd.read_csv(datasets["twitter"], encoding="utf-8", delimiter="‎", engine="python", quoting=QUOTE_NONE)
stream_of_consciousness_df = pd.read_csv(datasets["stream_of_consciousness"], encoding="utf-8", delimiter="‎", engine="python", quoting=QUOTE_NONE)

In [153]:
# Set label of school shooter or not
# 0 = NOT school shooter
# 1 = school shooter

schoolshootersinfo_df["label"] = 1
#manifesto_df["shooter"] = 1
stair_twitter_archive_df["label"] = 1
twitter_df["label"] = 0
stream_of_consciousness_df["label"] = 0

In [154]:
from word_embeddings import preprocess_text, get_glove_word_vectors

In [155]:
shooter_df = pd.concat([schoolshootersinfo_df, stair_twitter_archive_df], ignore_index=True)[:100]
non_shooter_df = pd.concat([twitter_df[:100], stream_of_consciousness_df[:100]])
whole_corpus_df = pd.concat([shooter_df, non_shooter_df], ignore_index=True).sample(frac=1)

In [156]:
whole_corpus_df["text"] = whole_corpus_df["text"].map(lambda a: get_glove_word_vectors(preprocess_text(a)))

c:\Users\Jonas\NTNU-Masters\src\experiments\5_language_models\word_embeddings.py:69: MarkupResemblesLocatorWarning: The input looks more like a filename than markup. You may want to open this file and pass the filehandle into Beautiful Soup.
  soup = BeautifulSoup(text, "html.parser")


In [157]:
from sklearn.model_selection import train_test_split 

In [158]:
shapes = [list(t.shape) for t in whole_corpus_df["text"]]
sentence_lengths = [dims[0] for dims in shapes]
max_sent_len = max(sentence_lengths)

def pad_embeddings(embedding_tensor, sentence_length):
    if len(embedding_tensor) < sentence_length:
        req_padding = sentence_length - embedding_tensor.shape[0]
        pad_tensor = torch.zeros(req_padding, emb_dim)
        embedding_tensor = torch.cat((embedding_tensor, pad_tensor), dim=0)
    
    return embedding_tensor

whole_corpus_df["text"] = whole_corpus_df["text"].map(lambda a: pad_embeddings(a, max_sent_len))

In [159]:
x_train, x_test, y_train, y_test = train_test_split(whole_corpus_df["text"], whole_corpus_df["label"], test_size=0.1)
x_train, x_val, y_train, y_val = train_test_split(x_train, y_train, test_size=0.1)

In [160]:
# Check sizes of tensors to shape input to mlp model
""" shapes = [list(t.shape) for t in whole_corpus_df["text"]]
dic_size = [dims[0] for dims in shapes]
print(shapes)


max_dic_size = max(dic_size)
min_dic_size = min(dic_size)
print(min_dic_size)
print(max_dic_size)
for x in x_train:
    if x.shape[0] == min_dic_size:
        print("x_shape: ", x.shape)
        i = 0
        for _ in x:
            print(_)
            i += 1
            print("i: ", i) """

' shapes = [list(t.shape) for t in whole_corpus_df["text"]]\ndic_size = [dims[0] for dims in shapes]\nprint(shapes)\n\n\nmax_dic_size = max(dic_size)\nmin_dic_size = min(dic_size)\nprint(min_dic_size)\nprint(max_dic_size)\nfor x in x_train:\n    if x.shape[0] == min_dic_size:\n        print("x_shape: ", x.shape)\n        i = 0\n        for _ in x:\n            print(_)\n            i += 1\n            print("i: ", i) '

In [161]:
# Creating datasets for use with dataloaders

from torch.utils.data import DataLoader, Dataset

class TextDataset(Dataset):
    def __init__(self, embeddings, labels, train=False):
        self.embeddings = embeddings
        self.labels = labels

    def __len__(self):
        if len(self.labels) == len(self.embeddings):
            return len(self.labels)
        else:
            return -1

    def __getitem__(self, idx):
        #print("embeddings: \n", self.embeddings[idx])
        #print("labels: \n", self.labels[idx])

        return self.embeddings[idx], self.labels[idx]

train_set = TextDataset(x_train.to_numpy(), y_train.to_numpy())
val_set = TextDataset(x_val.to_numpy(), y_val.to_numpy())
test_set = TextDataset(x_test.to_numpy(), y_test.to_numpy())

print(train_set.embeddings[3])

tensor([[ 0.0000,  0.0000,  0.0000,  ...,  0.0000,  0.0000,  0.0000],
        [ 0.3147,  0.4166,  0.1348,  ..., -0.2053,  0.0701, -0.1157],
        [-0.9321,  0.1596,  1.0410,  ..., -0.1744, -0.7260,  0.7140],
        ...,
        [ 0.0000,  0.0000,  0.0000,  ...,  0.0000,  0.0000,  0.0000],
        [ 0.0000,  0.0000,  0.0000,  ...,  0.0000,  0.0000,  0.0000],
        [ 0.0000,  0.0000,  0.0000,  ...,  0.0000,  0.0000,  0.0000]])


In [162]:
train_loader = DataLoader(train_set, batch_size=4, shuffle=True)
val_loader = DataLoader(val_set, batch_size=4, shuffle=False)

""" feats, label = next(iter(train_loader))
feats, label = next(iter(train_loader))

print(feats, label) """

' feats, label = next(iter(train_loader))\nfeats, label = next(iter(train_loader))\n\nprint(feats, label) '

In [163]:
# design model

import torch
import torch.nn as nn
import torch.nn.functional as F

device = "cuda" if torch.cuda.is_available() else "cpu"


emb_dim = 50
out_dim = 1

class TextClassifier(nn.Module):
    def __init__(self):
        super(TextClassifier, self).__init__()
        self.linear1 = nn.Linear(emb_dim, 100, device=device)
        self.linear2 = nn.Linear(100, 50, device=device)
        self.linear3 = nn.Linear(50, out_dim, device=device)

    def forward(self, x):
        x = self.linear1(x)
        x = F.relu(x)
        x = self.linear2(x)
        x = F.relu(x)
        x = self.linear3(x)
        x = F.sigmoid(x)

        return x

""" model = nn.Sequential(
    nn.Linear(emb_dim, 100, device=device),
    nn.ReLU(),
    nn.Linear(100, 50, device=device),
    nn.ReLU(),
    nn.Linear(50, out_dim, device=device),
    nn.Sigmoid()
) """

model = TextClassifier()
loss_fn = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

In [164]:
# Set up training steps and epoch 

def run_epoch(epoch_i, tb_writer):
    running_loss = 0.
    last_loss = 0.

    for i, data in enumerate(train_loader):
        inputs, labels = data
        #print(i, inputs, labels)
        optimizer.zero_grad()

        outputs = model(inputs)
        loss = loss_fn(outputs, labels)
        loss.backwards()

        optimizer.step()

        running_loss += loss.item()
        if i % 1000 == 999:
            last_loss = running_loss / 1000
            print(f"batch {i+1} loss: {last_loss}")
            #tb_x = epoch_i * len(train_loader) + i + 1
            #tb_writer.add_scalar("Loss/train", last_loss, tb_x)
            running_loss = 0.
        
    return last_loss

In [165]:
# Initializing in a separate cell so we can easily add more epochs to the same run
from datetime import datetime
from torch.utils.tensorboard import SummaryWriter

timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
#writer = SummaryWriter('runs/mlp_shooters_{}'.format(timestamp))
epoch_number = 0

EPOCHS = 5

best_vloss = 1_000_000.

for epoch in range(EPOCHS):
    print('EPOCH {}:'.format(epoch_number + 1))

    # Make sure gradient tracking is on, and do a pass over the data
    model.train(True)
    avg_loss = epoch(epoch_number, writer)

    # We don't need gradients on to do reporting
    model.train(False)

    running_vloss = 0.0
    for i, vdata in enumerate(validation_loader):
        vinputs, vlabels = vdata
        voutputs = model(vinputs)
        vloss = loss_fn(voutputs, vlabels)
        running_vloss += vloss

    avg_vloss = running_vloss / (i + 1)
    print('LOSS train {} valid {}'.format(avg_loss, avg_vloss))

    # Log the running loss averaged per batch
    # for both training and validation
    """ writer.add_scalars('Training vs. Validation Loss',
                    { 'Training' : avg_loss, 'Validation' : avg_vloss },
                    epoch_number + 1)
    writer.flush() """

    # Track best performance, and save the model's state
    if avg_vloss < best_vloss:
        best_vloss = avg_vloss
        model_path = 'model_{}_{}'.format(timestamp, epoch_number)
        torch.save(model.state_dict(), model_path)

    epoch_number += 1

EPOCH 1:


TypeError: 'int' object is not callable

In [ ]:
epochs = 20

for n in range(num_epochs):
    y_pred = model(X)
    loss = loss_fn(y_pred, y)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()